<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
!pip -q install duckdb huggingface_hub

In [21]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face secret created successfully")

Hugging Face secret created successfully


In [22]:
rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis + time window

One row represents the daily performance of **one content item for one client on one report date** in the `fact_content_daily_performance` table.

I use **March 2026** as the working time window because it is a middle month of the warehouse panel and is safer for feature development than the final month.

The analysis is intended to support **content refresh prioritization** using historical search and analytics signals that are available before the decision date.


In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Features

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_pageviews`
* `ga4_sessions`

These fields are historical measurements that would be available before a content refresh decision is made.

### Label / proxy

The intended output is a **refresh-priority ranking** for content items. No label-derived field is used as a feature.

### Context

* `client_hash_id`
* `content_hash_id`
* `report_date`

These fields identify observations and are used for grouping, filtering, and interpretation only.

### Excluded

* `client_has_gsc`
* `client_has_ga4`

These fields describe data availability and are treated as contextual metadata rather than predictive features.


In [24]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

The following checks verify the contract assumptions:

1. **Grain check** — confirm that one row corresponds to one client, one content item, and one report date.
2. **Availability check** — count rows where `gsc_data_available IS TRUE`.
3. **Missing-value check** — measure the share of missing values for key search and analytics fields.

### Planned feature frame

| Feature            | Available when?                                                     |
| ------------------ | ------------------------------------------------------------------- |
| `gsc_impressions`  | Historical search impressions available before the refresh decision |
| `gsc_clicks`       | Historical clicks available before the refresh decision             |
| `gsc_avg_position` | Historical search ranking available before the refresh decision     |
| `ga4_pageviews`    | Historical analytics data available before the refresh decision     |
| `ga4_sessions`     | Historical session data available before the refresh decision       |

### Leakage note

Any feature derived from future outcomes or directly from the target would be excluded because it could create **target leakage** and produce unrealistically high model performance.


In [25]:
# Query 1: grain check
grain = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS c
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

# Query 2: usable rows
usable = con.sql(f"""
SELECT COUNT(*) AS usable_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

# Query 3: missing values
missing = con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_clicks,
    AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS missing_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print("Grain check")
print(grain)

print("\nUsable rows")
print(usable)

print("\nMissing values")
print(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []

Usable rows
   usable_rows
0      3611061

Missing values
   missing_clicks  missing_sessions
0             0.0           0.30674


## 4. Data limits

* Clients may have different lengths of historical data.
* Some observations may not have GSC or GA4 data available.
* Missing values may represent unavailable measurements rather than zero activity.
* This notebook uses a **single monthly slice (March 2026)**, so the observations should be interpreted as a partial view of the warehouse rather than a complete historical evaluation.
* The analysis is intended for **decision support** and prioritization, not for establishing causal conclusions.


In [26]:
con.sql(f"""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY 1,2
ORDER BY rows DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available,rows
0,False,False,4690323
1,True,False,1718348
2,True,<NA>,1528366
3,False,<NA>,1490375
4,True,True,364347
5,False,True,49619


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.